In [115]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree,export_text
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.model_selection import GridSearchCV

OPIS SKUPA PODATAKA

The dataset consists of 10 numerical and 8 categorical attributes.
The 'Revenue' attribute can be used as the class label.

"Administrative", "Administrative Duration", "Informational", "Informational Duration", "Product Related" and "Product Related Duration" represent the number of different types of pages visited by the visitor in that session and total time spent in each of these page categories. The values of these features are derived from the URL information of the pages visited by the user and updated in real time when a user takes an action, e.g. moving from one page to another. The "Bounce Rate", "Exit Rate" and "Page Value" features represent the metrics measured by "Google Analytics" for each page in the e-commerce site. The value of "Bounce Rate" feature for a web page refers to the percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. The value of "Exit Rate" feature for a specific web page is calculated as for all pageviews to the page, the percentage that were the last in the session. The "Page Value" feature represents the average value for a web page that a user visited before completing an e-commerce transaction. The "Special Day" feature indicates the closeness of the site visiting time to a specific special day (e.g. Mother’s Day, Valentine's Day) in which the sessions are more likely to be finalized with transaction. The value of this attribute is determined by considering the dynamics of e-commerce such as the duration between the order date and delivery date. For example, for Valentina’s day, this value takes a nonzero value between February 2 and February 12, zero before and after this date unless it is close to another special day, and its maximum value of 1 on February 8. The dataset also includes operating system, browser, region, traffic type, visitor type as returning or new visitor, a Boolean value indicating whether the date of the visit is weekend, and month of the year.

In [116]:
exel = []

In [117]:
def metrika(title,preds,y_test):
    print(title, accuracy_score(y_test,preds))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))
    print("\nClassification Report:")
    print(classification_report(y_test, preds))
    print("-------------------------------------------------------------------")

In [118]:
def plot_roc_curve(fpr, tpr, roc_auc, model_name, set_name):

    output_image = 'slike2/'+model_name+'+'+ set_name+'.png'
    plt.figure(figsize=(8,6))
    plt.plot(fpr, tpr, color='blue', label=f'{model_name} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--',)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    # plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(output_image, dpi=300)
    plt.close()

In [119]:
def metrika1(model_name,y_pred,y_true,fpr,tpr,roc_auc,set_name):
    plot_roc_curve(fpr,tpr,roc_auc,model_name,set_name)
    return {
        "model": model_name,
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 3),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 3),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 3),
        "f1_score": round(float(f1_score(y_true, y_pred, average="weighted", zero_division=0)), 3),
        "f1_score_false": round(float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)), 3),
        "f1_score_true": round(float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)), 3)
    }

In [120]:
def parametri(result):
    with open("najbolji_parametri.txt", "a") as f:
        # for result in results:
            f.write(f"Model: {result['Model']}\n")
            f.write(f"Podaci: {result['Podaci']}\n")
            f.write(f"Best CV AUC: {result['Best CV AUC']:.4f}\n")
            f.write("Best Parameters:\n")
            
            for param, value in result["Best Params"].items():
                f.write(f"   {param}: {value}\n")
            
            f.write("-" * 40 + "\n")

In [121]:
def predicting(model, title, X_train, y_train, X_test, y_test,set_name,params):

    grid = GridSearchCV(estimator=model,param_grid=params,cv=5,n_jobs=-1)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_cv_score = grid.best_score_

    result_dict = {
        "Model": best_model,
        "Podaci": set_name,
        "Best CV AUC": best_cv_score,
        "Best Params": best_params
    }
    parametri(result_dict)

    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    preds = best_model.predict(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    met = metrika1(title,preds ,y_test,fpr,tpr,roc_auc,set_name)
    exel.append(met)
    print(met)

In [122]:
def predicting1(model, title, X_train, y_train, X_test, y_test,set_name,params):
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    # importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    # print (importances)
    met = metrika1(title,preds ,y_test,fpr,tpr,roc_auc,set_name)
    exel.append(met)
    print(met)

In [123]:
df = pd.read_csv(r'podaci\online+shoppers+purchasing+intention+dataset\online_shoppers_intention preprocessed.csv', encoding='cp1252', sep=',')
print(df.columns)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')


In [124]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,2,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,2,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,2,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,2,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,2,1,0


In [125]:
X = df.copy(deep=True)

In [126]:
X = X.drop("Revenue", axis=1)
y = df["Revenue"]

In [127]:
best5 = ["PageValues","ProductRelated_Duration","BounceRates","ExitRates","ProductRelated"]
best10 = ["PageValues","ProductRelated_Duration","BounceRates","ExitRates","ProductRelated","Administrative_Duration","Month","Administrative","Region","TrafficType","Informational_Duration"]
feature_names = X.columns

In [128]:
numeric_features = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues"
]

In [129]:
pca = PCA(n_components=2)
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

In [130]:
X_log = X.copy(deep=True)

In [131]:
X_log[numeric_features] = np.log1p(X_log[numeric_features])     # Logaritamsko skaliranje
X_sca_std = scaler_standard.fit_transform(X)                    # Standard scaler
X_sca_mm = scaler_minmax.fit_transform(X)                       # MinMax scaler
X_pca = pca.fit_transform(X)                                    # Samo PCA

X_pca_sca_std = pca.fit_transform(X_sca_std)                    # Standard scaler + PCA
X_pca_sca_mm = pca.fit_transform(X_sca_mm)                      # MinMax scaler + PCA

X_best5 =scaler_minmax.fit_transform(X[best5])
X_best10 = scaler_minmax.fit_transform(X[best10])

In [132]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123, stratify=y)

X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std = train_test_split(X_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm = train_test_split(X_sca_mm, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=123, stratify=y)

X_train_pca_sca_std, X_test_pca_sca_std, y_train_pca_sca_std, y_test_pca_sca_std = train_test_split(X_pca_sca_std, y, test_size=0.2, random_state=123, stratify=y)
X_train_pca_sca_mm, X_test_pca_sca_mm, y_train_pca_sca_mm, y_test_pca_sca_mm = train_test_split(X_pca_sca_mm, y, test_size=0.2, random_state=123, stratify=y)

X_train_best5, X_test_best5, y_train_best5, y_test_best5 = train_test_split(X_best5, y, test_size=0.2, random_state=123, stratify=y)
X_train_best10, X_test_best10, y_train_best10, y_test_best10 = train_test_split(X_best10, y, test_size=0.2, random_state=123, stratify=y)

In [133]:
datasets = [
    # ("Originalni podaci", X_train, X_test, y_train, y_test),
    # ("BEST5 podaci", X_train_best5, X_test_best5, y_train_best5, y_test_best5),
    ("BEST10 podaci", X_train_best10, X_test_best10, y_train_best10, y_test_best10),
    
    # ("Log transformacija", X_train_log, X_test_log, y_train_log, y_test_log),
    # ("StandardScaler", X_train_sca_std, X_test_sca_std, y_train_sca_std, y_test_sca_std),
    # ("MinMaxScaler", X_train_sca_mm, X_test_sca_mm, y_train_sca_mm, y_test_sca_mm),
    # ("PCA", X_train_pca, X_test_pca, y_train_pca, y_test_pca),

    # ("StandardScaler + PCA", X_train_pca_sca_std, X_test_pca_sca_std, y_train_pca_sca_std, y_test_pca_sca_std),
    # (" MinMaxScaler + PCA", X_train_pca_sca_mm, X_test_pca_sca_mm, y_train_pca_sca_mm, y_test_pca_sca_mm),
]

PREDVIDJANJE

In [134]:
dt_params = {
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"]
}

In [135]:
dt = DecisionTreeClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(dt, f"Stablo odlucivanja", X_tr, y_tr, X_te, y_te,name,dt_params)

{'model': 'Stablo odlucivanja', 'accuracy': 0.893, 'precision': 0.688, 'recall': 0.565, 'f1_score': 0.889, 'f1_score_false': 0.938, 'f1_score_true': 0.621}


In [136]:
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [137]:
rf = RandomForestClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(rf, f"Random Forest", X_tr, y_tr, X_te, y_te,name,rf_params)

{'model': 'Random Forest', 'accuracy': 0.904, 'precision': 0.74, 'recall': 0.589, 'f1_score': 0.9, 'f1_score_false': 0.944, 'f1_score_true': 0.656}


In [138]:
mlp_params = {
    "hidden_layer_sizes": [
        (50,),
        (100,),
        (50, 50),
        (100, 50)
    ],
    "activation": ["relu", "tanh"],
    "solver": ["adam"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate": ["constant", "adaptive"],
    "max_iter": [500]
}

In [139]:
mlp = MLPClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(mlp, f"MLP", X_tr, y_tr, X_te, y_te,name,mlp_params)

c:\Users\Korisnik\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\ma\core.py:2881: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


{'model': 'MLP', 'accuracy': 0.899, 'precision': 0.692, 'recall': 0.628, 'f1_score': 0.897, 'f1_score_false': 0.941, 'f1_score_true': 0.658}


In [140]:
nb_params = {
    "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
}

In [141]:
nb = GaussianNB()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(nb, f"NB", X_tr, y_tr, X_te, y_te,name,nb_params)

{'model': 'NB', 'accuracy': 0.833, 'precision': 0.471, 'recall': 0.644, 'f1_score': 0.843, 'f1_score_false': 0.898, 'f1_score_true': 0.544}


In [142]:
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

In [143]:
knn = KNeighborsClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(knn, f"KNN", X_tr, y_tr, X_te, y_te,name,knn_params)

{'model': 'KNN', 'accuracy': 0.876, 'precision': 0.735, 'recall': 0.312, 'f1_score': 0.854, 'f1_score_false': 0.93, 'f1_score_true': 0.438}


In [144]:
lr_params = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
    "class_weight": [None, "balanced"],
    "max_iter": [1000]
}

In [145]:
lr = LogisticRegression()

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(lr, f"LR", X_tr, y_tr, X_te, y_te,name,lr_params)

{'model': 'LR', 'accuracy': 0.888, 'precision': 0.766, 'recall': 0.395, 'f1_score': 0.872, 'f1_score_false': 0.936, 'f1_score_true': 0.522}


In [146]:
svm_params = {
    "C": [0.1, 1], 
    "kernel": ["rbf"],  
    "gamma": ['scale', 00.1]
}

In [147]:
svm = SVC(probability=True)

for name, X_tr, X_te, y_tr, y_te in datasets:
    predicting(svm, f"SVM", X_tr, y_tr, X_te, y_te,name,svm_params)

{'model': 'SVM', 'accuracy': 0.892, 'precision': 0.801, 'recall': 0.401, 'f1_score': 0.876, 'f1_score_false': 0.939, 'f1_score_true': 0.534}


In [148]:
xgb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

In [149]:
name1=''

In [150]:
xgb = XGBClassifier()

for name, X_tr, X_te, y_tr, y_te in datasets:
    name1 = name
    predicting(xgb, f"XGB", X_tr, y_tr, X_te, y_te,name,xgb_params)

{'model': 'XGB', 'accuracy': 0.91, 'precision': 0.737, 'recall': 0.647, 'f1_score': 0.907, 'f1_score_false': 0.947, 'f1_score_true': 0.689}


In [151]:
# for name, X_tr, X_te, y_tr, y_te in datasets:
#     name1 = name

#     y_train_oh = to_categorical(y_tr)
#     y_test_oh = to_categorical(y_te)

#     model = Sequential([
#         Dense(128, input_shape=(X_tr.shape[1],), activation='relu'),
#         Dropout(0.3),
#         Dense(64, activation='relu'),
#         Dropout(0.3),
#         Dense(32, activation='relu'),
#         # Dropout(0.3),
#         Dense(16, activation='relu'),
#         # Dropout(0.3),
#         Dense(8, activation='relu'),
#         Dense(2, activation='softmax')
#     ])

#     model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#     model.fit(X_tr, y_train_oh, epochs=50, batch_size=8, verbose=0)

#     y_pred_prob = model.predict(X_te)
#     y_pred = y_pred_prob.argmax(axis=1)

#     y_prob = y_pred_prob[:, 1]
#     fpr, tpr, thresholds = roc_curve(y_test, y_prob)
#     roc_auc = roc_auc_score(y_test, y_prob)

#     # plot_roc_curve(fpr,tpr,roc_auc,'NN',name)
#     met = metrika1('NN',y_pred , y_te,fpr,tpr,roc_auc,name)
#     exel.append(met)

In [152]:
df_exel = pd.DataFrame(exel)
df_exel.to_excel(f"rezKlasifikacije2/rezultati_{name1}.xlsx", index=False)